# 01 — Data, sample definition and target

**Goal:** turn 2.2M raw LendingClub loans into a clean modelling dataset with a
defensible default definition — and avoid the two classic traps: **data leakage**
and **resolution bias**.

Before running: put the Kaggle file in `data/raw/` (see README).

In [ ]:
import os, sys, pickle
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pdmodel import config as C
pd.set_option("display.float_format", "{:.4f}".format)
PROC = ROOT / "data" / "processed"
from pdmodel import data, plots

## 1. Load only what we need
The raw file has 151 columns. We read 24: information known **at application time**,
plus `loan_status` (the outcome) and `grade` / `sub_grade` / `int_rate` (benchmark only, never model inputs).

In [ ]:
DATA_PATH = os.environ.get("PD_DATA_PATH")   # None -> auto-find in data/raw/
NROWS = None                                   # e.g. 300_000 for a quick first run
raw = data.load_raw(DATA_PATH, nrows=NROWS)
print(raw.shape)
raw.head()

In [ ]:
raw["loan_status"].value_counts()

## 2. Target definition
| `loan_status` | Treatment |
|---|---|
| Fully Paid | good (0) |
| Charged Off, Default | bad (1) |
| Current, Late, In Grace Period | **excluded** — outcome not yet known |

**Why only 36-month loans issued ≤ 2015?** The data snapshot ends in 2018Q4. A loan
issued in 2017 that has already *finished* is more likely to be an early default
(or early prepayment) than a typical loan — keeping it would bias the default rate.
Restricting to vintages that have had 36 months to mature avoids this.

In [ ]:
df = data.prepare(raw)
print(f"{len(df):,} loans after sample definition, default rate {df['default'].mean():.2%}")
df.head()

## 3. Default rate by vintage
If later vintages default more, a model built on old vintages will **under-predict**
PD on new ones. The out-of-time test in notebook 04 checks exactly this.

In [ ]:
plots.default_rate_by_vintage(df); plt.show()
df.groupby("issue_year")["default"].agg(n="size", default_rate="mean")

## 4. Train / test / out-of-time split
- **Train / test**: random 70/30 split of vintages ≤ 2013 (stratified on default).
- **OOT**: vintages 2014–2015 — the model never sees them.

A bank validator cares most about the OOT numbers: they show how the model
behaves on *future* borrowers.

In [ ]:
S = data.split_samples(df)
data.sample_summary(S)

## 5. Missing values
WoE binning gives missing values their own bin, so we don't need to impute.
But check *why* data is missing — e.g. `mort_acc` is missing for early vintages
because LendingClub only started collecting it later.

In [ ]:
S["train"][C.NUMERIC_FEATURES + C.CATEGORICAL_FEATURES].isna().mean().sort_values(ascending=False).head(8)

In [ ]:
PROC.mkdir(parents=True, exist_ok=True)
pd.to_pickle(S, PROC / "samples.pkl")
print("saved", PROC / "samples.pkl")

## ✍️ Check your understanding (interview prep)
1. Why is `total_pymnt` not allowed as a feature, even though it predicts default almost perfectly?
2. Why exclude `int_rate` and `grade`? When *would* it be fine to include them?
3. What is resolution bias, and how does the 2015 cut-off avoid it?
4. Our PD is a *lifetime (36m)* PD. How does that differ from the 12-month PD used in Basel / IFRS 9?